## Overview

Anomaly detection is the process of **identifying data points or patterns that deviate significantly from the majority of the data**. 

We may search for outliers because we may want to remove them in order to run better machine learning algorithms, or because we want to analyze them as they may represent interesting phenomena (es. fraud detection, network security, fault detection etc...).

We may approach the problem:
- in **supervised way**, if we have labeled data (i.e., we know which points are normal and which are anomalies). In this case, we know the labels of the anomalies and we can train a classification model to detect them. It is rare to have labeled data for anomalies, because anomalies are rare by definition.
- **unsupervised way**, if we don't have labeled data. It is what we usually work on. In this case, we can use clustering or density-based methods to identify points that are far from the majority of the data. 

## Types of Anomalies
We can have:
- **Point Anomalies**: A point anomaly is a single data point that is significantly different from the rest of the data. For example, in a dataset of temperatures, a temperature reading of 1000 degrees Celsius would be a point anomaly.
<img src="img_teoria/point_anomaly.png" alt="point anomaly" width="300px"/>

- **Contextual Anomalies**: A contextual anomaly is a data point that is considered anomalous in a specific context but may be normal in another context. For example, a dog picture that has the feature tipically associated with cats pictures may be considered a contextual anomaly. Or in the context of temperatures, a temperature of 30 degrees Celsius may be normal in summer but anomalous in winter.
<img src="img_teoria/contextual_anomaly.png" alt="contextual anomaly" width="300px"/>

- **Collective Anomalies**: A collective anomaly is a group of data points that are anomalous when considered together, but may not be anomalous individually. ex. if a single person deposits 9000$ and sends it to an off-shore account, it may not be anomalous, but if a group of people do the same thing in a short period of time, it may be considered a collective anomaly (potential fraud).
<img src="img_teoria/collective_anomlay.png" alt="collective anomaly" width="300px"/>


## Techniques for Anomaly Detection
There are many families of techniques for anomaly detection. All of them are functions that look at the data points and assign an anomaly score to each point, indicating how likely it is to be an anomaly. Then we can set a threshold on the anomaly score to classify points as normal or anomalous.

In scikit-learn we have the parameter "contamination" that indicates the proportion of anomalies in the dataset. Below are some of the most common techniques:

- **Statistical Methods**: These methods assume that the data follows a certain distribution (es. Gaussian) and identify points that deviate significantly from this distribution. Examples include Z-score, Chi-square test.
- **Clustering-Based Methods**: These methods use clustering algorithms (es. K-Means, DBSCAN) to group similar data points together and identify points that do not belong to any cluster or belong to small clusters as anomalies.
- **Proximity-Based Methods**: These methods measure the distance or density of data points and identify points that are far from other points or in low-density regions as anomalies. Examples include k-Nearest Neighbors (k-NN), Local Outlier Factor (LOF).
- **Tree-Based Methods**: These methods use decision trees or random forests to model the normal behavior of the data and identify points that do not fit this model as anomalies. Examples include Isolation Forest.
- **Boundary-Based Methods**: These methods use decision boundaries to separate normal points from anomalies. Examples include One-Class SVM.
- **Neural Network-Based Methods**: These methods use neural networks to learn the normal patterns in the data and identify points that do not conform to these patterns as anomalies. Examples include Autoencoders.

### Statistical Methods
Imagine to want to detect anomalies in terms of height of people. We can assume that the heights of people follow a normal distribution (Gaussian distribution). We can compute the mean and standard deviation of the heights in the dataset, and then we can compute the Z-score for each height. The Z-score measures how many standard deviations a point is from the mean. If a point has a Z-score greater than a certain threshold (es. 3), we can consider it an anomaly. $z-score = x-media/std$

<img src="img_teoria/statistical_anomaly.png" alt="statistical anomaly" width="400px"/>

### Cluster-Based Methods
In clustering-based methods:

- **K-Means** to cluster the data points. After clustering, we can compute the distance of each point from its cluster centroid. Points that are far from their centroid (greater than a certain threshold) can be considered anomalies.
<p>
<img src="img_teoria/kmeansss.png" alt="kmeans anomaly" width="45%"/>
<img src="img_teoria/kmeans22.png" alt="kmeans anomaly 2" width="45%"/>
</p>

- **DBSCAN** to cluster the data points. DBSCAN naturally can identify points that do not belong to any cluster (its *noise points*) as anomalies. So each noise point identified by DBSCAN can be considered an anomaly.

### Proximity-Based Methods (LOF)
**LOF Local Outlier Factor** considers a point to be anomalous if it lives in a low-density region *compared to its neighbors*. In fact, differently from DBSCAN, LOF addresses the problem of identifying anomalies in datasets with varying density. 

Since in fact different regions of space can have different densities, LOF estimates the "local" density of each region of space, by looking at the density of the k-nearest neighbors of each point. **If a point has a significantly lower density than its (k) neighbors, it is considered an anomaly.**

We'll simplify a bit the algorithm for the lecture. For a given point, we compute its **local density**, which is *inversely proportional* to avg of the distance to its k-nearest neighbors (infatti se la media delle distanze dei suoi ad esempio k=5 vicini è alta, significa che il punto è lontano dai suoi vicini, quindi la densità locale è bassa). (**Large Distance -> Low Density** while **Small Distance -> High Density**).

Now if we assumed that all high density regions of space had the same density, we could simply define a threshold on the local density to identify anomalies (points with local density below the threshold are anomalies).  But as mentioned we could have regions of space with different densities, so we need a more flexible approach so that for dense regions we consider a smaller threshold (since the outlier points for a very dense region will have a anyway a high local density) and for sparse regions we consider a larger threshold (since the outlier points will have a lower density). In the example we see how setting a small threshold would identify the red anomaly for the dense region, but identify also as anomalies some normal points in the sparse region. 

<img src="img_teoria/lof1.png" alt="lof anomaly" width="400px"/>

**So the idea is the following**. We estimate the local density of a point (red) and the avg of the local densities of its k-nearest neighbors (orange, dark blue and green). If the local density of the point is significantly lower than the avg of the local densities of its neighbors, then we consider it an anomaly. (NB facciamo riferimento alla densità ma in pratica calcoliamo le distanze medie dai vicini, che sono inversamente proporzionali alla densità, quindi se la distanza del punto dalla distanza media dei vicini è circa la stessa allora anche la densità è circa la stessa).

NB. Per prima cosa ci calcoliamo il raggio del punto rosso necessario a coprire i k (3 nell'esempio) vicini, e poi la media dei tre raggi dei vicini (sempre per coprire tre vicini). Quindi in pratica stiamo calcolando il raggio e facciamo riferimento ad esso per capire la densità (questo perchè densità è inversamente proporzionale alla distanza media dai vicini).

In the example, the red point has pretty much the same density as the avg of its neighbors, so it is not considered an anomaly.

<p>
<img src="img_teoria/lof2.png" alt="lof anomaly 2" width="60%"/>
<img src="img_teoria/lof3.png" alt="lof anomaly 3" width="25%"/>
</p>

If we consider on the other hand an anomalous point (red), it will have a much lower density (higher radius to cover three neighbors) than the avg of its neighbors because those neighbors live in a dense region , so the red point will be considered an anomaly.

<img src="img_teoria/lof4.png" alt="lof anomaly 4" width="400px"/>

So by using this approach we can identify anomalies in datasets with varying density!

### Tree-Based Methods (Isolation Forest)
A **Isolation Forest** is a set of trees that are not anymore trained like decision tree, but are randomly generated. With "randomly generated" we mean the following: while with normal decision trees we select the feature and the threshold that best splits the data at each node (based for example on the GINI Index), with Isolation Trees we randomly select a feature and a split value to split the data at each node.

After having created a bunch of trees that split randomly the data, I will say that I have an **Anomaly** if it was a point that was isolated in fewer splits (**short root-to-leaf path length**). I will declare **Normal Points** instead as points that require more splits to be isolated.

In fact, since anomalies are rare and different from normal points, they will be isolated quickly in the random splits, while normal points will require more splits to be isolated because they are nearer to each other. (Remember in fact that each split in the tree represents a split in the feature space)

In the example below, x0 is an anomaly and in fact is isolated very quickly!

<img src="img_teoria/isolation_forest.png" alt="isolation forest" width="400px"/>

To remove some of the stocasticity of the method, we create many trees (forest) and average the results. Then we compute on average how hard is it to isolate each point across all trees, and we set a threshold on this value to classify points as normal or anomalous.

The **anomaly score** of a point can be computed as:

$Anomaly_score(x) = 2^{-\frac{E(h(x))}{c(n)}}$

where E(h(x)) is the average path length of point x across all trees (how many splits are needed to isolate x on average), and c(n) is a normalization factor that depends on the number of points n in the dataset. c(n) is needed because the average path length depends on the number of points in the dataset (more points -> deeper trees on average -> more splits needed to isolate a point)

This algorithm is quite naive and works well to find anomalies that are very distant from normal points, but may fail to find anomalies that are closer to normal points. In the example below the points in the middle are anomalies, but they are close to normal points, so they may are not isolated quickly and so considered as normal points.

<img src="img_teoria/isolation_forest_2.png" alt="isolation forest 2" width="300px"/>

### Boundary-Based Methods (One-Class SVM)
We'll focus on this method in other courses.

## Neural Network-Based Methods (Autoencoders)
Autoencoders are neural networks.
The idea is to train a neural network to reconstruct the input data. We could simply do it with a single layer and identity activation function, but what we actually do here is to force the network to compress the input n-dimensional data into a lower-dimensional representation (called "code") (**encoder part**) and then reconstruct the original data from this compressed representation (**decoder part**).

<img src="img_teoria/autoencoder.png" alt="autoencoder" width="400px"/>

By doing the encoder part, we are forcing the model to learn only the most important features of the data, so that it can reconstruct the original data from a lower-dimensional representation. (example imagine that you have a page to write a compressed version of a book, from which you can reconstruct the original book as accurately as possible).

In all this, **anomalies are the elements that are reconstructed poorly**. In fact, since anomalies are rare and different from normal points, they don't follow common patterns in the data, so the autoencoder will have a hard time reconstructing them accurately.

**So in practice we take the original and the reconstructed inputs, we use a loss function that tells us how different they are (ex. the loss function MSE) and we set a threshold on this value to classify points as normal or anomalous.**